In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [ ]:
# Parser for df: KokkosComm benchmarks only (excludes raw_benchmark_ entries).
# Reads _mean and _stddev rows from the new repetition-based output format.

def parse_benchmark_file(filename):
    ansi_escape = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')
    rma_map = {
        'lock_unlock_put':        'LockUnlockPut',
        'lock_unlock_get':        'LockUnlockGet',
        'lock_unlock_accumulate': 'LockUnlockAccumulate',
        'fence_put':              'FencePut',
        'fence_get':              'FenceGet',
        'fence_accumulate':       'FenceAccumulate',
        'pscw_put':               'PSCWPut',
        'pscw_get':               'PSCWGet',
        'pscw_accumulate':        'PSCWAccumulate',
        'sendrecv_comparison':    'SendRecv',
    }
    # Anchored to ^ so raw_benchmark_ lines never match
    mean_re   = re.compile(r'^benchmark_(\w+)/(\d+)/manual_time_mean\s+([\d.]+)\s+us.*bytes_per_second=([\d.]+)(Ki|Mi|Gi)/s')
    stddev_re = re.compile(r'^benchmark_(\w+)/(\d+)/manual_time_stddev\s+([\d.]+)\s+us.*bytes_per_second=([\d.]+)(Ki|Mi|Gi)/s')

    def to_gis(val, unit):
        if unit == 'Ki': return val / (1024.0 * 1024.0)
        if unit == 'Mi': return val / 1024.0
        return val

    data = {}
    with open(filename, 'r') as f:
        for line in f:
            cl = ansi_escape.sub('', line).strip()
            m = mean_re.search(cl)
            if m:
                op = rma_map.get(m.group(1))
                if op:
                    key = (op, int(m.group(2)))
                    data.setdefault(key, {}).update({
                        'Operation':      op,
                        'MessageSize':    int(m.group(2)),
                        'Time_us':        float(m.group(3)),
                        'Throughput_Gis': to_gis(float(m.group(4)), m.group(5)),
                    })
            s = stddev_re.search(cl)
            if s:
                op = rma_map.get(s.group(1))
                if op:
                    key = (op, int(s.group(2)))
                    data.setdefault(key, {}).update({
                        'Stddev_us':        float(s.group(3)),
                        'Stddev_Gis':       to_gis(float(s.group(4)), s.group(5)),
                    })

    rows = [v for v in data.values() if 'Time_us' in v]
    return pd.DataFrame(rows)

df = parse_benchmark_file('results.txt')
print(f"Loaded {len(df)} benchmarks")
print("Operations:", sorted(df['Operation'].unique()))

In [ ]:
df

In [ ]:
# Shared color, marker, and label maps used by all plots below
operations = df['Operation'].unique()

colors = {
    'LockUnlockPut':        '#0066CC',
    'LockUnlockGet':        '#0099FF',
    'LockUnlockAccumulate': '#003399',
    'FencePut':             '#CC0000',
    'FenceGet':             '#FF4444',
    'FenceAccumulate':      '#990000',
    'PSCWPut':              '#AA00AA',
    'PSCWGet':              '#FF00FF',
    'PSCWAccumulate':       '#660066',
    'SendRecv':             '#00AA00',
}

markers = {
    'LockUnlockPut':        'o',
    'LockUnlockGet':        'D',
    'LockUnlockAccumulate': 's',
    'FencePut':             's',
    'FenceGet':             'p',
    'FenceAccumulate':      'h',
    'PSCWPut':              'v',
    'PSCWGet':              '<',
    'PSCWAccumulate':       '>',
    'SendRecv':             '^',
}

label_map = {
    'LockUnlockPut':        'Lock/Unlock (Put)',
    'LockUnlockGet':        'Lock/Unlock (Get)',
    'LockUnlockAccumulate': 'Lock/Unlock (Accumulate)',
    'FencePut':             'Fence (Put)',
    'FenceGet':             'Fence (Get)',
    'FenceAccumulate':      'Fence (Accumulate)',
    'PSCWPut':              'PSCW (Put)',
    'PSCWGet':              'PSCW (Get)',
    'PSCWAccumulate':       'PSCW (Accumulate)',
    'SendRecv':             'SendRecv (Baseline)',
}

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for op in operations:
    d = df[df['Operation'] == op].sort_values('MessageSize')
    yerr = d['Stddev_us'] if 'Stddev_us' in d.columns else None
    ax.errorbar(d['MessageSize'], d['Time_us'], yerr=yerr,
                marker=markers.get(op, 'o'), linewidth=2.5, markersize=8,
                label=label_map.get(op, op), color=colors.get(op, '#000000'),
                capsize=3, elinewidth=1)

ax.set_xscale('log')
ax.set_xlabel('Message Size (bytes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Latency (microseconds)', fontsize=14, fontweight='bold')
ax.set_title('RMA Synchronization Method Latency Comparison', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('latency_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graph saved as: latency_comparison.png")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for op in operations:
    d = df[df['Operation'] == op].sort_values('MessageSize')
    yerr = d['Stddev_Gis'] if 'Stddev_Gis' in d.columns else None
    ax.errorbar(d['MessageSize'], d['Throughput_Gis'], yerr=yerr,
                marker=markers.get(op, 'o'), linewidth=2.5, markersize=8,
                label=label_map.get(op, op), color=colors.get(op, '#000000'),
                capsize=3, elinewidth=1)

ax.set_xscale('log')
ax.set_xlabel('Message Size (bytes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Throughput (Gi/s)', fontsize=14, fontweight='bold')
ax.set_title('RMA Throughput Comparison', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('throughput_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graph saved as: throughput_comparison.png")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

small_sizes = [1, 8, 64, 512]
large_sizes = [4096, 32768, 262144]

for op in operations:
    d_small = df[(df['Operation'] == op) & (df['MessageSize'].isin(small_sizes))].sort_values('MessageSize')
    yerr = d_small['Stddev_us'] if 'Stddev_us' in d_small.columns else None
    ax1.errorbar(d_small['MessageSize'], d_small['Time_us'], yerr=yerr,
                 marker=markers.get(op, 'o'), linewidth=2.5, markersize=10,
                 label=label_map.get(op, op), color=colors.get(op, '#000000'),
                 capsize=3, elinewidth=1)

    d_large = df[(df['Operation'] == op) & (df['MessageSize'].isin(large_sizes))].sort_values('MessageSize')
    yerr2 = d_large['Stddev_Gis'] if 'Stddev_Gis' in d_large.columns else None
    ax2.errorbar(d_large['MessageSize'], d_large['Throughput_Gis'], yerr=yerr2,
                 marker=markers.get(op, 'o'), linewidth=2.5, markersize=10,
                 label=label_map.get(op, op), color=colors.get(op, '#000000'),
                 capsize=3, elinewidth=1)

ax1.set_xscale('log')
ax1.set_xlabel('Message Size (bytes)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Latency (μs)', fontsize=13, fontweight='bold')
ax1.set_title('Small Messages (Latency-Bound)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.set_xscale('log')
ax2.set_xlabel('Message Size (bytes)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Throughput (Gi/s)', fontsize=13, fontweight='bold')
ax2.set_title('Large Messages (Bandwidth-Bound)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('small_vs_large_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graph saved as: small_vs_large_comparison.png")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for op in operations:
    d = df[df['Operation'] == op].copy().sort_values('MessageSize')
    d['Efficiency'] = d['Throughput_Gis'] / d['Time_us']
    ax.plot(d['MessageSize'], d['Efficiency'],
            marker=markers.get(op, 'o'), linewidth=3, markersize=10,
            label=label_map.get(op, op), color=colors.get(op, '#000000'))

ax.set_xscale('log')
ax.set_xlabel('Message Size (bytes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Efficiency (Gi/s per μs)', fontsize=14, fontweight='bold')
ax.set_title('Efficiency Metric: Throughput per Microsecond', fontsize=16, fontweight='bold')
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('efficiency_metric.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graph saved as: efficiency_metric.png")

In [ ]:
sendrecv_data = df[df['Operation'] == 'SendRecv'][['MessageSize', 'Time_us']].rename(columns={'Time_us': 'SendRecv_Time'})
normalized_df = df.merge(sendrecv_data, on='MessageSize')
normalized_df['Relative_Latency'] = normalized_df['Time_us'] / normalized_df['SendRecv_Time']

fig, ax = plt.subplots(figsize=(12, 7))

for op in operations:
    if op != 'SendRecv':
        d = normalized_df[normalized_df['Operation'] == op].sort_values('MessageSize')
        ax.plot(d['MessageSize'], d['Relative_Latency'],
                marker=markers.get(op, 'o'), linewidth=2.5, markersize=8,
                label=label_map.get(op, op), color=colors.get(op, '#000000'))

ax.axhline(y=1.0, color='#00AA00', linestyle='--', linewidth=3, label='SendRecv (Baseline)', alpha=0.7)
ax.set_xscale('log')
ax.set_xlabel('Message Size (bytes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Relative Latency (vs SendRecv)', fontsize=14, fontweight='bold')
ax.set_title('Performance Normalized to SendRecv Baseline', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.axhline(y=2.0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.text(1, 0.5, '2x faster', fontsize=10, alpha=0.6, va='bottom')
ax.text(1, 2.0, '2x slower', fontsize=10, alpha=0.6, va='bottom')

plt.tight_layout()
plt.savefig('normalized_performance.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graph saved as: normalized_performance.png")

## KokkosComm Wrapper Overhead: KokkosComm vs Raw MPI
The plots below show KokkosComm wrapper latency against raw MPI for each sync mode, with SendRecv as a baseline. Solid lines = KokkosComm, dashed lines = Raw MPI.

In [ ]:
# Extended parser: reads both benchmark_ (KokkosComm) and raw_benchmark_ (Raw MPI) entries.
# Prefixes Raw_ on raw entries. Also reads _mean and _stddev rows.
import re as _re
import pandas as _pd

_ansi = _re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')
_rma_map = {
    'lock_unlock_put':        'LockUnlockPut',
    'lock_unlock_get':        'LockUnlockGet',
    'lock_unlock_accumulate': 'LockUnlockAccumulate',
    'fence_put':              'FencePut',
    'fence_get':              'FenceGet',
    'fence_accumulate':       'FenceAccumulate',
    'pscw_put':               'PSCWPut',
    'pscw_get':               'PSCWGet',
    'pscw_accumulate':        'PSCWAccumulate',
    'sendrecv_comparison':    'SendRecv',
}
_mean_re   = _re.compile(r'(raw_)?benchmark_(\w+)/(\d+)/manual_time_mean\s+([\d.]+)\s+us.*bytes_per_second=([\d.]+)(Ki|Mi|Gi)/s')
_stddev_re = _re.compile(r'(raw_)?benchmark_(\w+)/(\d+)/manual_time_stddev\s+([\d.]+)\s+us.*bytes_per_second=([\d.]+)(Ki|Mi|Gi)/s')

def _to_gis(val, unit):
    if unit == 'Ki': return val / (1024.0 * 1024.0)
    if unit == 'Mi': return val / 1024.0
    return val

_data = {}
with open('results.txt', 'r') as _f:
    for _line in _f:
        _cl = _ansi.sub('', _line).strip()
        _m = _mean_re.search(_cl)
        if _m:
            _op = _rma_map.get(_m.group(2))
            if _op:
                _prefix = 'Raw_' if _m.group(1) else ''
                _key = (_prefix + _op, int(_m.group(3)))
                _data.setdefault(_key, {}).update({
                    'Operation':      _prefix + _op,
                    'MessageSize':    int(_m.group(3)),
                    'Time_us':        float(_m.group(4)),
                    'Throughput_Gis': _to_gis(float(_m.group(5)), _m.group(6)),
                })
        _s = _stddev_re.search(_cl)
        if _s:
            _op = _rma_map.get(_s.group(2))
            if _op:
                _prefix = 'Raw_' if _s.group(1) else ''
                _key = (_prefix + _op, int(_s.group(3)))
                _data.setdefault(_key, {}).update({
                    'Stddev_us':  float(_s.group(4)),
                    'Stddev_Gis': _to_gis(float(_s.group(5)), _s.group(6)),
                })

df_raw = _pd.DataFrame([v for v in _data.values() if 'Time_us' in v])
print(f"df_raw loaded: {len(df_raw)} rows")
print("Operations:", sorted(df_raw['Operation'].unique()))

In [ ]:
# Lock/Unlock: KokkosComm vs Raw MPI
fig, ax = plt.subplots(figsize=(12, 7))

_sr = df_raw[df_raw['Operation'] == 'SendRecv'].sort_values('MessageSize')
if not _sr.empty:
    ax.plot(_sr['MessageSize'], _sr['Time_us'], marker='^', linewidth=2, markersize=7,
            color='#00AA00', linestyle=':', label='SendRecv (baseline)', zorder=5)

for _kk_op, _raw_op, _color, _marker, _oplabel in [('LockUnlockPut', 'Raw_LockUnlockPut', '#0066CC', 'o', 'Put'), ('LockUnlockGet', 'Raw_LockUnlockGet', '#3399FF', 'D', 'Get'), ('LockUnlockAccumulate', 'Raw_LockUnlockAccumulate', '#003377', 's', 'Accumulate')]:
    _kk  = df_raw[df_raw['Operation'] == _kk_op].sort_values('MessageSize')
    _raw = df_raw[df_raw['Operation'] == _raw_op].sort_values('MessageSize')
    _kk_err  = _kk['Stddev_us']  if 'Stddev_us' in _kk.columns  else None
    _raw_err = _raw['Stddev_us'] if 'Stddev_us' in _raw.columns else None
    if not _kk.empty:
        ax.errorbar(_kk['MessageSize'], _kk['Time_us'], yerr=_kk_err,
                    marker=_marker, linewidth=2.5, markersize=8, color=_color,
                    linestyle='-', label=f'KokkosComm {_oplabel}', capsize=3, elinewidth=1)
    if not _raw.empty:
        ax.errorbar(_raw['MessageSize'], _raw['Time_us'], yerr=_raw_err,
                    marker=_marker, linewidth=1.8, markersize=6, color=_color,
                    linestyle='--', label=f'Raw MPI {_oplabel}', alpha=0.6, capsize=3, elinewidth=1)

ax.set_xscale('log')
ax.set_xlabel('Message Size (bytes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Latency (microseconds)', fontsize=14, fontweight='bold')
ax.set_title('Lock/Unlock: KokkosComm vs Raw MPI', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lock_unlock_kokkos_vs_raw.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graph saved as: lock_unlock_kokkos_vs_raw.png")

In [ ]:
# Fence: KokkosComm vs Raw MPI
fig, ax = plt.subplots(figsize=(12, 7))

_sr = df_raw[df_raw['Operation'] == 'SendRecv'].sort_values('MessageSize')
if not _sr.empty:
    ax.plot(_sr['MessageSize'], _sr['Time_us'], marker='^', linewidth=2, markersize=7,
            color='#00AA00', linestyle=':', label='SendRecv (baseline)', zorder=5)

for _kk_op, _raw_op, _color, _marker, _oplabel in [('FencePut', 'Raw_FencePut', '#CC2200', 'o', 'Put'), ('FenceGet', 'Raw_FenceGet', '#FF6644', 'D', 'Get'), ('FenceAccumulate', 'Raw_FenceAccumulate', '#881100', 's', 'Accumulate')]:
    _kk  = df_raw[df_raw['Operation'] == _kk_op].sort_values('MessageSize')
    _raw = df_raw[df_raw['Operation'] == _raw_op].sort_values('MessageSize')
    _kk_err  = _kk['Stddev_us']  if 'Stddev_us' in _kk.columns  else None
    _raw_err = _raw['Stddev_us'] if 'Stddev_us' in _raw.columns else None
    if not _kk.empty:
        ax.errorbar(_kk['MessageSize'], _kk['Time_us'], yerr=_kk_err,
                    marker=_marker, linewidth=2.5, markersize=8, color=_color,
                    linestyle='-', label=f'KokkosComm {_oplabel}', capsize=3, elinewidth=1)
    if not _raw.empty:
        ax.errorbar(_raw['MessageSize'], _raw['Time_us'], yerr=_raw_err,
                    marker=_marker, linewidth=1.8, markersize=6, color=_color,
                    linestyle='--', label=f'Raw MPI {_oplabel}', alpha=0.6, capsize=3, elinewidth=1)

ax.set_xscale('log')
ax.set_xlabel('Message Size (bytes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Latency (microseconds)', fontsize=14, fontweight='bold')
ax.set_title('Fence: KokkosComm vs Raw MPI', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fence_kokkos_vs_raw.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graph saved as: fence_kokkos_vs_raw.png")

In [ ]:
# PSCW: KokkosComm vs Raw MPI
fig, ax = plt.subplots(figsize=(12, 7))

_sr = df_raw[df_raw['Operation'] == 'SendRecv'].sort_values('MessageSize')
if not _sr.empty:
    ax.plot(_sr['MessageSize'], _sr['Time_us'], marker='^', linewidth=2, markersize=7,
            color='#00AA00', linestyle=':', label='SendRecv (baseline)', zorder=5)

for _kk_op, _raw_op, _color, _marker, _oplabel in [('PSCWPut', 'Raw_PSCWPut', '#AA00AA', 'o', 'Put'), ('PSCWGet', 'Raw_PSCWGet', '#DD44DD', 'D', 'Get'), ('PSCWAccumulate', 'Raw_PSCWAccumulate', '#660066', 's', 'Accumulate')]:
    _kk  = df_raw[df_raw['Operation'] == _kk_op].sort_values('MessageSize')
    _raw = df_raw[df_raw['Operation'] == _raw_op].sort_values('MessageSize')
    _kk_err  = _kk['Stddev_us']  if 'Stddev_us' in _kk.columns  else None
    _raw_err = _raw['Stddev_us'] if 'Stddev_us' in _raw.columns else None
    if not _kk.empty:
        ax.errorbar(_kk['MessageSize'], _kk['Time_us'], yerr=_kk_err,
                    marker=_marker, linewidth=2.5, markersize=8, color=_color,
                    linestyle='-', label=f'KokkosComm {_oplabel}', capsize=3, elinewidth=1)
    if not _raw.empty:
        ax.errorbar(_raw['MessageSize'], _raw['Time_us'], yerr=_raw_err,
                    marker=_marker, linewidth=1.8, markersize=6, color=_color,
                    linestyle='--', label=f'Raw MPI {_oplabel}', alpha=0.6, capsize=3, elinewidth=1)

ax.set_xscale('log')
ax.set_xlabel('Message Size (bytes)', fontsize=14, fontweight='bold')
ax.set_ylabel('Latency (microseconds)', fontsize=14, fontweight='bold')
ax.set_title('PSCW: KokkosComm vs Raw MPI', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pscw_kokkos_vs_raw.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graph saved as: pscw_kokkos_vs_raw.png")